# 🚕 NYC Yellow Taxi Real-Time Trip Analytics — Apache Flink + ClickHouse
**Source:** NYC TLC Trip Record Data (data.cityofnewyork.us — free, public)
**Stack:** NYC TLC API → Kafka → Apache Flink → ClickHouse (OLAP)
**Orchestration:** Apache Airflow

## Overview
Real-time streaming pipeline that simulates NYC Yellow Taxi trip events from the
NYC Open Data API, processes them with Apache Flink for windowed aggregations
(trip counts, fare averages, hotspot detection), and stores analytics in ClickHouse —
a columnar OLAP database optimized for real-time analytical queries.

**Data Source:** NYC TLC Trip Records via Socrata API
URL: https://data.cityofnewyork.us/resource/uacg-pexx.json
Free, no API key required. ~1M+ yellow taxi trips/month.

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests kafka-python apache-flink clickhouse-driver pandas

import requests
import json
import time
import logging
import pandas as pd
from datetime import datetime, timezone, timedelta, date
from kafka import KafkaProducer
import clickhouse_driver
from pyflink.datastream import StreamExecutionEnvironment, RuntimeExecutionMode
from pyflink.table import (
    StreamTableEnvironment, TableConfig, DataTypes, Schema
)
from pyflink.table.window import Tumble
from pyflink.common import WatermarkStrategy, Duration

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('NYCTaxiPipeline')
print('✅ All imports loaded')

In [ ]:
# NYC Open Data Socrata API — free, no auth needed
NYC_API_BASE     = 'https://data.cityofnewyork.us/resource'
NYC_TAXI_DATASET = 'uacg-pexx.json'   # Yellow Taxi Trip Records 2024
NYC_APP_TOKEN    = ''                  # optional, increases rate limit

# Kafka
KAFKA_BOOTSTRAP   = 'localhost:9092'
KAFKA_TOPIC_TRIPS = 'nyc-taxi-trips'

# ClickHouse
CLICKHOUSE_HOST   = 'localhost'
CLICKHOUSE_PORT   = 9000
CLICKHOUSE_DB     = 'nyc_taxi_analytics'
CLICKHOUSE_USER   = 'default'
CLICKHOUSE_PASS   = ''

# Payment type mapping
PAYMENT_TYPES = {
    '1': 'Credit card', '2': 'Cash', '3': 'No charge',
    '4': 'Dispute',     '5': 'Unknown', '6': 'Voided trip'
}

print('✅ Config ready')
print(f'   Kafka: {KAFKA_BOOTSTRAP} → {KAFKA_TOPIC_TRIPS}')
print(f'   ClickHouse: {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}/{CLICKHOUSE_DB}')

## Section 2 — NYC TLC API Producer → Kafka

In [ ]:
def fetch_nyc_taxi_trips(limit: int = 1000, offset: int = 0,
                          month: str = None) -> list:
    """
    Fetch Yellow Taxi trips from NYC Open Data Socrata API.
    Filter by month if specified (format: '2024-01').
    """
    url = f'{NYC_API_BASE}/{NYC_TAXI_DATASET}'
    params = {
        '$limit':  limit,
        '$offset': offset,
        '$order':  'tpep_pickup_datetime DESC',
    }
    if month:
        params['$where'] = (
            f"tpep_pickup_datetime >= '{month}-01T00:00:00' "
            f"AND tpep_pickup_datetime < '{month}-01T00:00:00' + interval'1 month'"
        )
    if NYC_APP_TOKEN:
        params['$$app_token'] = NYC_APP_TOKEN

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def serialize_trip(raw: dict) -> dict:
    """Normalize raw trip record to typed dict."""
    def safe_float(val, default=0.0):
        try: return float(val)
        except: return default
    def safe_int(val, default=0):
        try: return int(val)
        except: return default

    pickup_str  = raw.get('tpep_pickup_datetime',  '')
    dropoff_str = raw.get('tpep_dropoff_datetime', '')
    payment_id  = str(raw.get('payment_type', '5'))

    # Compute duration in minutes
    try:
        pickup_dt  = datetime.fromisoformat(pickup_str.replace('T', ' ').split('.')[0])
        dropoff_dt = datetime.fromisoformat(dropoff_str.replace('T', ' ').split('.')[0])
        duration_min = max(0, (dropoff_dt - pickup_dt).seconds // 60)
    except:
        duration_min = 0

    fare     = safe_float(raw.get('fare_amount'))
    tip      = safe_float(raw.get('tip_amount'))
    total    = safe_float(raw.get('total_amount'))
    distance = safe_float(raw.get('trip_distance'))

    return {
        'vendor_id':          str(raw.get('vendorid', '')),
        'pickup_datetime':    pickup_str,
        'dropoff_datetime':   dropoff_str,
        'passenger_count':    safe_int(raw.get('passenger_count'), 1),
        'trip_distance_miles':distance,
        'pu_location_id':     safe_int(raw.get('pulocationid'), 0),
        'do_location_id':     safe_int(raw.get('dolocationid'), 0),
        'payment_type_id':    safe_int(payment_id, 5),
        'payment_type_name':  PAYMENT_TYPES.get(payment_id, 'Unknown'),
        'fare_amount':        fare,
        'tip_amount':         tip,
        'total_amount':       total,
        'tip_rate_pct':       round(tip / fare * 100, 2) if fare > 0 else 0.0,
        'duration_minutes':   duration_min,
        'speed_mph':          round(distance / (duration_min / 60), 2) if duration_min > 0 else 0.0,
        'rate_code_id':       safe_int(raw.get('ratecodeid'), 1),
        'store_and_fwd_flag': raw.get('store_and_fwd_flag', 'N'),
        'congestion_surcharge': safe_float(raw.get('congestion_surcharge')),
        'airport_fee':        safe_float(raw.get('airport_fee')),
        'ingested_at':        datetime.now(timezone.utc).isoformat(),
        'source':             'nyc-tlc-open-data'
    }

def stream_taxi_to_kafka(total_records: int = 5000, batch_size: int = 500) -> int:
    """
    Fetch NYC taxi trips in batches and publish to Kafka,
    simulating a continuous stream.
    """
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    total_published = 0
    for offset in range(0, total_records, batch_size):
        try:
            raw_trips = fetch_nyc_taxi_trips(limit=batch_size, offset=offset)
            for raw in raw_trips:
                trip = serialize_trip(raw)
                producer.send(
                    KAFKA_TOPIC_TRIPS,
                    key=f"{trip['pickup_datetime']}_{trip['pu_location_id']}".encode(),
                    value=trip
                )
                total_published += 1
            producer.flush()
            logger.info(f'  Offset {offset}: published {len(raw_trips)} trips')
            time.sleep(0.5)
        except Exception as e:
            logger.warning(f'  Batch error at offset {offset}: {e}')

    producer.close()
    logger.info(f'✅ Total published: {total_published} taxi trips')
    return total_published

# stream_taxi_to_kafka(total_records=5000)

## Section 3 — Apache Flink Streaming Processing

In [ ]:
FLINK_JOB = '''
# Flink job definition (run as standalone PyFlink script)
# pyflink requires separate JVM — run in Flink cluster or docker

from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment
from pyflink.table.expressions import col, lit
from pyflink.table.window import Tumble

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(4)
t_env = StreamTableEnvironment.create(env)

# Define Kafka source
t_env.execute_sql("""
    CREATE TABLE kafka_taxi_trips (
        vendor_id           STRING,
        pickup_datetime     TIMESTAMP(3),
        dropoff_datetime    TIMESTAMP(3),
        passenger_count     INT,
        trip_distance_miles DOUBLE,
        pu_location_id      INT,
        do_location_id      INT,
        payment_type_name   STRING,
        fare_amount         DOUBLE,
        tip_amount          DOUBLE,
        total_amount        DOUBLE,
        tip_rate_pct        DOUBLE,
        duration_minutes    INT,
        speed_mph           DOUBLE,
        congestion_surcharge DOUBLE,
        WATERMARK FOR pickup_datetime AS pickup_datetime - INTERVAL 10 SECOND
    ) WITH (
        'connector'                     = 'kafka',
        'topic'                         = 'nyc-taxi-trips',
        'properties.bootstrap.servers'  = 'localhost:9092',
        'properties.group.id'           = 'flink-taxi-consumer',
        'format'                        = 'json',
        'scan.startup.mode'             = 'latest-offset'
    )
""")

# Define ClickHouse sink
t_env.execute_sql("""
    CREATE TABLE clickhouse_trip_agg (
        window_start        TIMESTAMP(3),
        window_end          TIMESTAMP(3),
        pu_location_id      INT,
        trip_count          BIGINT,
        avg_fare            DOUBLE,
        avg_tip_rate        DOUBLE,
        avg_duration_min    DOUBLE,
        total_revenue       DOUBLE,
        avg_speed_mph       DOUBLE,
        credit_card_pct     DOUBLE
    ) WITH (
        'connector'  = 'jdbc',
        'url'        = 'jdbc:clickhouse://localhost:8123/nyc_taxi_analytics',
        'table-name' = 'trip_5min_agg',
        'driver'     = 'com.clickhouse.jdbc.ClickHouseDriver'
    )
""")

# 5-minute tumbling window aggregation
t_env.execute_sql("""
    INSERT INTO clickhouse_trip_agg
    SELECT
        TUMBLE_START(pickup_datetime, INTERVAL 5 MINUTE) AS window_start,
        TUMBLE_END(pickup_datetime,   INTERVAL 5 MINUTE) AS window_end,
        pu_location_id,
        COUNT(*)                                          AS trip_count,
        ROUND(AVG(fare_amount), 2)                       AS avg_fare,
        ROUND(AVG(tip_rate_pct), 2)                      AS avg_tip_rate,
        ROUND(AVG(duration_minutes), 1)                  AS avg_duration_min,
        ROUND(SUM(total_amount), 2)                      AS total_revenue,
        ROUND(AVG(speed_mph), 1)                         AS avg_speed_mph,
        ROUND(SUM(CASE WHEN payment_type_name = \'Credit card\' THEN 1 ELSE 0 END)
              / COUNT(*) * 100, 1)                       AS credit_card_pct
    FROM TABLE(
        TUMBLE(TABLE kafka_taxi_trips, DESCRIPTOR(pickup_datetime), INTERVAL 5 MINUTE)
    )
    GROUP BY TUMBLE(pickup_datetime, INTERVAL 5 MINUTE), pu_location_id
""")
'''
print('📋 Flink job SQL defined')
print('   Run as: flink run -py flink_taxi_job.py')

## Section 4 — ClickHouse Schema & Analytics

In [ ]:
def get_clickhouse_client():
    return clickhouse_driver.Client(
        host=CLICKHOUSE_HOST, port=CLICKHOUSE_PORT,
        database=CLICKHOUSE_DB, user=CLICKHOUSE_USER,
        password=CLICKHOUSE_PASS
    )

CLICKHOUSE_DDL = """
-- Raw trips table (MergeTree engine — columnar, fast aggregations)
CREATE TABLE IF NOT EXISTS nyc_taxi_analytics.taxi_trips (
    vendor_id            String,
    pickup_datetime      DateTime,
    dropoff_datetime     DateTime,
    passenger_count      UInt8,
    trip_distance_miles  Float32,
    pu_location_id       UInt16,
    do_location_id       UInt16,
    payment_type_name    String,
    fare_amount          Float32,
    tip_amount           Float32,
    total_amount         Float32,
    tip_rate_pct         Float32,
    duration_minutes     UInt16,
    speed_mph            Float32,
    congestion_surcharge Float32,
    ingested_at          DateTime DEFAULT now()
) ENGINE = MergeTree()
PARTITION BY toYYYYMM(pickup_datetime)
ORDER BY (pickup_datetime, pu_location_id)
SETTINGS index_granularity = 8192;

-- 5-minute aggregation table (written by Flink)
CREATE TABLE IF NOT EXISTS nyc_taxi_analytics.trip_5min_agg (
    window_start     DateTime,
    window_end       DateTime,
    pu_location_id   UInt16,
    trip_count       UInt32,
    avg_fare         Float32,
    avg_tip_rate     Float32,
    avg_duration_min Float32,
    total_revenue    Float32,
    avg_speed_mph    Float32,
    credit_card_pct  Float32
) ENGINE = ReplacingMergeTree(window_start)
ORDER BY (window_start, pu_location_id);
"""

def setup_clickhouse_schema():
    client = get_clickhouse_client()
    client.execute(f'CREATE DATABASE IF NOT EXISTS {CLICKHOUSE_DB}')
    for stmt in [s.strip() for s in CLICKHOUSE_DDL.split(';') if s.strip() and not s.startswith('--')]:
        try:
            client.execute(stmt)
        except Exception as e:
            logger.warning(f'DDL: {e}')
    logger.info('✅ ClickHouse schema ready')

def run_clickhouse_analytics():
    client = get_clickhouse_client()

    # Top 10 busiest pickup zones last hour
    result = client.execute("""
        SELECT pu_location_id,
               count()           AS trip_count,
               avg(fare_amount)  AS avg_fare,
               avg(tip_rate_pct) AS avg_tip_pct
        FROM taxi_trips
        WHERE pickup_datetime >= now() - INTERVAL 1 HOUR
        GROUP BY pu_location_id
        ORDER BY trip_count DESC
        LIMIT 10
    """)
    df = pd.DataFrame(result, columns=['location_id','trips','avg_fare','avg_tip_pct'])
    print('\n🗺️  Top 10 busiest pickup zones (last 1h):')
    print(df.to_string(index=False))
    return df

# setup_clickhouse_schema()
# run_clickhouse_analytics()

## Section 5 — Pipeline Summary

| Layer | Technology | Details |
|---|---|---|
| **Source** | NYC TLC Open Data (Socrata API) | Yellow Taxi trips, free, no key |
| **Ingest** | Apache Kafka | `nyc-taxi-trips`, GZIP |
| **Stream Processing** | **Apache Flink** | 5-min tumbling windows, watermark 10s |
| **OLAP Storage** | **ClickHouse** | MergeTree engine, columnar, < 100ms queries |
| **Orchestration** | Apache Airflow | Hourly data fetch + Flink job trigger |

**Why ClickHouse?** MergeTree columnar engine handles billions of rows with sub-second
aggregation queries — ideal for real-time taxi analytics dashboards.

**Why Flink over Spark?** Lower latency event processing, native SQL windowing,
and better support for exactly-once semantics in streaming.